# Q8.
```{admonition}
:class: note
This problem involves the `OJ` data set.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
oj = pd.read_csv('../../ALL CSV FILES - 2nd Edition/OJ.csv')
oj['Store7'] = oj['Store7'] == 'Yes'

## (a)
```{admonition}
:class: note
Create a training set containing a random sample of $800$ observations, and a test set containing the remaining observations.

In [3]:
X, y = oj.drop(columns='Purchase'), oj['Purchase']
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=800,random_state=1728)

In [4]:
oj.sample(5)

,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
921,CH,269,2,1.86,2.18,0.0,0.0,0,0,0.616000,2.18,1.86,0.32,False,0.0,0.000000,0.32,2
880,CH,264,4,1.99,2.09,0.1,0.0,0,0,0.767142,2.09,1.89,0.20,False,0.0,0.050251,0.10,4
279,MM,242,3,1.99,2.23,0.0,0.0,0,0,0.007206,2.23,1.99,0.24,False,0.0,0.000000,0.24,3
619,CH,267,7,1.86,2.13,0.0,0.0,1,0,0.866588,2.13,1.86,0.27,True,0.0,0.000000,0.27,0
116,CH,257,2,1.86,2.18,0.0,0.0,0,0,0.900647,2.18,1.86,0.32,False,0.0,0.000000,0.32,2


In [5]:
cat_feats = ['StoreID','SpecialMM','SpecialCH','Store7','STORE']
numeric_feats = [col for col in X.columns if col not in cat_feats]

## (b)
```{admonition}
:class: note
Fit a support vector classifier to the training data using `C = 0.01`, with `Purchase` as the response and the other variables as predictors. How many support points are there?

In [6]:
ct = ColumnTransformer(
    transformers=[
        ('scaling', StandardScaler(),numeric_feats),
        ('one_hot', OneHotEncoder(),cat_feats)
    ]
)

cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=1728)
errors = []

In [7]:
pipe_svc1 = Pipeline(
    [
        ('transform',ct),
        ('svc_linear',SVC(kernel='linear', C=0.01))
    ]
)

pipe_svc1.fit(X_train,y_train)
support_points = pipe_svc1.named_steps['svc_linear'].support_vectors_
print(f'There are {len(support_points)} support points')

There are 440 support points


## (c)
```{admonition}
:class: note
What are the training and test error rates?

In [8]:
svcl_001_train_err = 1-accuracy_score(pipe_svc1.predict(X_train),y_train)
svcl_001_test_err = 1-accuracy_score(pipe_svc1.predict(X_test),y_test)
errors.append(['Linear Base',svcl_001_test_err])
print(f'Train error rate is {svcl_001_train_err:.4f}')
print(f'Test error rate is {svcl_001_test_err:.4f}')

Train error rate is 0.1625
Test error rate is 0.1667


## (d)
```{admonition}
:class: note
Use cross-validation to select an optimal `C`. Consider values in the range $0.01$ to $10$.

In [9]:
Cs = np.logspace(-2,1,100)

pipe_svcl_gen = Pipeline(
    [
        ('transform',ct),
        ('svc_linear',SVC(kernel='linear'))
    ]
)

grid_svcl = GridSearchCV(pipe_svcl_gen,param_grid={'svc_linear__C':Cs},cv=cv,n_jobs=-1,scoring='accuracy')
grid_svcl.fit(X_train,y_train)
best_lin_c = grid_svcl.best_params_['svc_linear__C']
print(f'Best C value is {best_lin_c:.4f}')

Best C value is 0.0107


## (e)
```{admonition}
:class: note
Compute the training and test error rates using this new value for `C`.

In [10]:
best_lin_model = grid_svcl.best_estimator_

svcl_best_train_err = 1-accuracy_score(best_lin_model.predict(X_train),y_train)
svcl_best_test_err = 1-accuracy_score(best_lin_model.predict(X_test),y_test)
errors.append(['Linear CV',svcl_best_test_err])
print(f'Train error rate is {svcl_best_train_err:.4f}')
print(f'Test error rate is {svcl_best_test_err:.4f}')

Train error rate is 0.1650
Test error rate is 0.1667


## (f)
```{admonition}
:class: note
Repeat parts (b) through (e) using a support vector machine with a radial kernel. Use the default value for `gamma`.

In [11]:
pipe_rbf = Pipeline(
    [
        ('transform',ct),
        ('svc_rbf',SVC(kernel='rbf', C=0.01))
    ]
)

pipe_rbf.fit(X_train,y_train)
rbf_support_points = pipe_rbf.named_steps['svc_rbf'].support_vectors_
print(f'There are {len(rbf_support_points)} support points')

There are 619 support points


In [12]:
rbf_train_err = 1-accuracy_score(pipe_rbf.predict(X_train),y_train)
rbf_test_err = 1-accuracy_score(pipe_rbf.predict(X_test),y_test)
errors.append(['Radial Base',rbf_test_err])
print(f'Train error rate is {rbf_train_err:.4f}')
print(f'Test error rate is {rbf_test_err:.4f}')

Train error rate is 0.3850
Test error rate is 0.4037


In [13]:
pipe_rbf_gen = Pipeline(
    [
        ('transform',ct),
        ('svc_rbf',SVC(kernel='rbf'))
    ]
)

grid_rbf = GridSearchCV(pipe_rbf_gen,param_grid={'svc_rbf__C':Cs},cv=cv,n_jobs=-1,scoring='accuracy')
grid_rbf.fit(X_train,y_train)
best_rbf_c = grid_rbf.best_params_['svc_rbf__C']
print(f'Best C value is {best_rbf_c:.4f}')

Best C value is 1.1498


In [14]:
best_rbf_model = grid_rbf.best_estimator_

svc_rbf_best_train_err = 1-accuracy_score(best_rbf_model.predict(X_train),y_train)
svc_rbf_best_test_err = 1-accuracy_score(best_rbf_model.predict(X_test),y_test)
errors.append(['Radial CV',svc_rbf_best_test_err])
print(f'Train error rate is {svc_rbf_best_train_err:.4f}')
print(f'Test error rate is {svc_rbf_best_test_err:.4f}')

Train error rate is 0.1500
Test error rate is 0.1852


## (g)
```{admonition}
:class: note
Repeat parts (b) through (e) using a support vector machine with a polynomial kernel. Set `degree = 2`.

In [15]:
pipe_poly = Pipeline(
    [
        ('transform',ct),
        ('svc_poly',SVC(kernel='poly', C=0.01,degree=2))
    ]
)

pipe_poly.fit(X_train,y_train)
poly_support_points = pipe_poly.named_steps['svc_poly'].support_vectors_
print(f'There are {len(poly_support_points)} support points')

There are 621 support points


In [16]:
poly_train_err = 1-accuracy_score(pipe_poly.predict(X_train),y_train)
poly_test_err = 1-accuracy_score(pipe_poly.predict(X_test),y_test)
errors.append(['Quadratic Base',poly_test_err])
print(f'Train error rate is {poly_train_err:.4f}')
print(f'Test error rate is {poly_test_err:.4f}')

Train error rate is 0.3750
Test error rate is 0.3852


In [17]:
pipe_poly_gen = Pipeline(
    [
        ('transform',ct),
        ('svc_poly',SVC(kernel='poly',degree=2))
    ]
)

grid_poly = GridSearchCV(pipe_poly_gen,param_grid={'svc_poly__C':Cs},cv=cv,n_jobs=-1,scoring='accuracy')
grid_poly.fit(X_train,y_train)
best_poly_c = grid_poly.best_params_['svc_poly__C']
print(f'Best C value is {best_poly_c:.4f}')

Best C value is 2.0092


In [18]:
best_poly_model = grid_poly.best_estimator_

svc_poly_best_train_err = 1-accuracy_score(best_poly_model.predict(X_train),y_train)
svc_poly_best_test_err = 1-accuracy_score(best_poly_model.predict(X_test),y_test)
errors.append(['Quadratic CV',svc_poly_best_test_err])
print(f'Train error rate is {svc_poly_best_train_err:.4f}')
print(f'Test error rate is {svc_poly_best_test_err:.4f}')

Train error rate is 0.1512
Test error rate is 0.1926


## (h)
```{admonition}
:class: note
Overall, which approach seems to give the best results on this data?

In [20]:
lowest_test_error = min(errors,key=lambda x: x[1])
print(f'Lowest test error was {lowest_test_error[1]:.3f} using {lowest_test_error[0]}.')

Lowest test error was 0.167 using Linear Base.
